# MindFlow Step 4b · Which hand features matter?

**Question.** Of the 38 hand + SPY numbers that HGB (and the Step 4 hybrid) use, which carry the volatility signal and which are redundant?


Every choice is made on a **selection year** (2022, the last training year) with HGB fitted on 2016–2021; 2023 is used
once, to confirm the chosen subsets next to the full model (paired day bootstrap). The test years are not used.

1. **Permutation importance**: AUC drop on 2022 when one feature is shuffled (5 repeats).
2. **Feature groups**: stock returns (16), stock volatility (2), stock volume and time of day (2), SPY (12),
   stock vs SPY (6): retrain without each group, and with each group alone.
3. **Forward selection**: add the feature that raises 2022 AUC most, until within 0.002 of all 38.
4. **Confirmation on 2023**: full, forward-selected, top-k by importance and each group-drop model.

CPU is enough for this notebook (no GPU needed).


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing) |
| **7** | Efficient computing for the final model: 16-bit / INT8 inference, LoRA fine-tuning of a pretrained ViT | `Step7a_Efficient_Computing` |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import chartcnn
import feature_select as fs
print("feature_select", fs.__version__)
DEVICE = torch.device("cpu")

## 2. Configuration
**Time (estimate, Colab CPU):** permutation importance ~5 min per cell; group ablation ~5 min; forward selection over the
top `forward_candidates` features ~30–60 min per cell. `QUICK = True` uses fewer rows and 10 candidates.

In [ ]:
SMOKE = LOCAL_TEST
RUN_ID = None
QUICK = False
HORIZONS = [1, 12]                                 # bars of 5 minutes
SETTINGS = dict(stride=12, selection_start="2022-01-01", embargo_sessions=2,
                max_inner=300_000, max_selection=200_000, max_train=1_000_000,
                n_repeats=5, tolerance=0.002, max_features=15, forward_candidates=20, top_k=(5, 10), seed=69)
if QUICK:
    SETTINGS.update(max_inner=100_000, max_selection=60_000, max_train=300_000, forward_candidates=10, n_repeats=3)
if SMOKE:
    SETTINGS.update(selection_start=None, max_inner=2_000, max_selection=2_000, max_train=3_000, n_repeats=1,
                    forward_candidates=4, max_features=3)
CELLS = [dict(name=f"vol_h{h}", kind="vol", target="vol", horizon=h, minutes=5 * h, stride=SETTINGS["stride"]) for h in HORIZONS]
RUN_ID, RUN_DIR, RESUME = mf.new_run(HERE, "4b", RUN_ID, "features", LOCAL_TEST)
store = chartcnn.Store(PREPARED)
if SETTINGS["selection_start"] is None:             # synthetic data: last ~third of the training period
    tr_days = np.unique(store.dates[store.dates <= store.config["train_end"]])
    SETTINGS["selection_start"] = str(tr_days[int(len(tr_days) * 0.65)])
mf.save_settings(RUN_DIR, RESUME, settings=SETTINGS, cells=CELLS, quick=QUICK, feature_select=fs.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DIR, "| selection year starts", SETTINGS["selection_start"])
pd.DataFrame([dict(group=g, n=len(c), features=", ".join(c)) for g, c in fs.GROUPS.items()]).set_index("group")

## 3. Selection on 2022, confirmation on 2023, per cell

In [ ]:
ALL = {}
for cell in CELLS:
    out = RUN_DIR / cell["name"]; out.mkdir(exist_ok=True)
    if (out / "confirmation.csv").exists():
        print(cell["name"], "already done"); ALL[cell["name"]] = {k: pd.read_csv(out / f"{k}.csv") for k in
                                                                  ("importance", "groups", "forward", "confirmation")}
        continue
    print("\n" + "=" * 20, cell["name"], "=" * 20)
    t0 = time.perf_counter()
    d, X = fs.build_data(store, cell, SETTINGS)
    yi, ys = d["inner"].label.to_numpy(), d["sel"].label.to_numpy()
    full, full_auc = fs.fit_score(list(X["inner"].columns), X["inner"], yi, X["sel"], ys, SETTINGS["seed"])
    print(f"all 38 features: selection AUC {full_auc:.4f} ({fs.tree_size(full):,} tree nodes)")
    base, imp = fs.permutation_importance(full, X["sel"], ys, SETTINGS["n_repeats"], SETTINGS["seed"])
    fs.plot_importance(imp, out / "importance.png", f"{cell['name']}: permutation importance (selection year)")
    print("groups (retrained on inner train, scored on the selection year):")
    groups = fs.group_ablation(X["inner"], yi, X["sel"], ys, SETTINGS["seed"], full_auc)
    cand = list(imp.feature[:SETTINGS["forward_candidates"]])
    print(f"forward selection over the top {len(cand)} features by importance:")
    chosen, fwd = fs.forward_selection(X["inner"], yi, X["sel"], ys, SETTINGS["seed"], full_auc, SETTINGS["tolerance"],
                                       SETTINGS["max_features"], cand)
    subsets = {"all 38 features": list(X["train"].columns), f"forward-selected ({len(chosen)})": chosen}
    for k in SETTINGS["top_k"]:
        subsets[f"top {k} by importance"] = list(imp.feature[:k])
    for g, gcols in fs.GROUPS.items():
        subsets[f"without {g}"] = [c for c in X["train"].columns if c not in gcols]
        subsets[f"only {g}"] = [c for c in gcols if c in X["train"].columns]
    conf, _ = fs.confirm(subsets, d, X, SETTINGS["seed"], n_boot=50 if SMOKE else 300)
    res = dict(importance=imp, groups=groups, forward=fwd, confirmation=conf)
    for k, v in res.items():
        v.to_csv(out / f"{k}.csv", index=False)
    ALL[cell["name"]] = res
    print(f"{cell['name']} done in {(time.perf_counter() - t0) / 60:.1f} min")

## 4. Results
Confirmation on 2023: AUC with 95% day-bootstrap interval, and the paired difference to the full 38-feature model
("tied" = the subset loses nothing measurable).

In [ ]:
for name, r in ALL.items():
    print(f"=== {name}: top 15 by permutation importance (selection year)"); display(r["importance"].head(15).round(5))
    print("groups"); display(r["groups"].round(4))
    print("forward selection path"); display(r["forward"].round(4))
    print("confirmation on validation 2023"); display(r["confirmation"].round(4))
useless = {name: set(r["importance"].loc[r["importance"].auc_drop <= 0, "feature"]) for name, r in ALL.items()}
common = set.intersection(*useless.values()) if useless else set()
print("features with no importance (AUC drop ≤ 0) in every cell:", ", ".join(sorted(common)) or "none")

### How to read it
- **Forward-selected subset tied with all 38 on 2023**: the rest are redundant for HGB; a smaller feature branch can be
  used in the Step 4 hybrid.
- **Without a group, AUC falls (worse)**: that group carries information the others do not.
- **Only a group ≈ all 38**: one group (likely the stock's own volatility) carries most of the signal.
- Permutation importance understates correlated features (shuffling one leaves its twin); the group retraining and the
  forward path are the checks for that.